# 08D_B_Skill_Classification

Classify remaining skills using Gold Taxonomy -> Fuzzy Match -> NLP Fallback.

In [1]:
import pandas as pd
from rapidfuzz import process, fuzz
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

c:\Users\Saanvi\anaconda3\envs\COURSE\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Data

In [2]:
skills = pd.read_csv('../Generated Datasets/linkedin_skill_universe_1000.csv')
gold = pd.read_csv('../Generated Datasets/taxonomy_gold_247.csv')

skills['skill'] = skills['skill'].astype(str).str.lower().str.strip()
gold['skill'] = gold['skill'].astype(str).str.lower().str.strip()

## Exact Match Dictionary

In [3]:
gold_map = {}
for _, row in gold.iterrows():
    gold_map[row['skill']] = (
        row['category'],
        row['sub_category']
    )

## NLP Setup

In [4]:
model = SentenceTransformer('all-MiniLM-L6-v2')

label_text = {
    'Technical':'programming software engineering cloud database ai analytics devops cybersecurity',
    'Soft Skill':'communication leadership teamwork interpersonal collaboration adaptability',
    'Domain Knowledge':'healthcare finance sales marketing hr education operations legal management',
    'Certification':'certification license credential professional qualification',
    'Language':'english spanish french german hindi language proficiency',
    'Noise':'benefits insurance degree diploma employment condition'
}

label_names = list(label_text.keys())
label_emb = model.encode(list(label_text.values()), normalize_embeddings=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6357.16it/s]


## Classification Engine

In [5]:
gold_skills = list(gold_map.keys())
results = []

for skill in skills['skill']:

    if skill in gold_map:
        cat, sub = gold_map[skill]
        results.append([cat, sub, 1.00, 'Exact'])
        continue

    fuzzy = process.extractOne(skill, gold_skills, scorer=fuzz.token_sort_ratio)

    if fuzzy and fuzzy[1] >= 92:
        matched_skill = fuzzy[0]
        cat, sub = gold_map[matched_skill]
        results.append([cat, sub, round(fuzzy[1]/100,2), 'Fuzzy'])
        continue

    emb = model.encode([skill], normalize_embeddings=True)
    sims = cosine_similarity(emb, label_emb)[0]

    idx = np.argmax(sims)
    conf = float(sims[idx])
    category = label_names[idx]

    if conf < 0.55:
        results.append(['Review','Review',conf,'NLP_Review'])
    else:
        results.append([category, category, conf, 'NLP'])

In [6]:
skills[['category','sub_category','confidence','classification_source']] = pd.DataFrame(results)

## Save Final Master Dataset

In [7]:
freq_col = [c for c in skills.columns if c not in ['skill','category','sub_category','confidence','classification_source']][0]

final_df = skills[[
    'skill',
    freq_col,
    'category',
    'sub_category',
    'confidence'
]].copy()

final_df.columns = [
    'skill',
    'linkedin_frequency',
    'category',
    'sub_category',
    'confidence'
]

final_df.to_csv('../Generated Datasets/expanded_skill_master.csv', index=False)

print(final_df['category'].value_counts())

category
Review              2320
Domain Knowledge     131
Noise                 54
Soft Skill            52
Technical             31
Certification         21
Language               8
Name: count, dtype: int64
